# Wenu planisphere — explicit drawing version

This version draws every component explicitly instead of relying on `CelestialSphere.draw()`.
The filled horizon disk is forced behind every astronomical layer.


In [ ]:
import inspect
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from skyfield.api import Loader

import wenu
print(f"Wenu version: {wenu.__version__}")
from wenu.observer import Observer
from wenu.projection import StereographicProjection
from wenu.renderers import layers
from wenu.sky import CelestialSphere
from wenu.resources import catalog_path, constellation_lines_path, boundary_path


In [ ]:

MAGNITUDE_LIMIT=6.0
PROJECTION_RADIUS=2.0
FLIP_EAST_WEST=True
#SKY_COLOR="slateblue"
SKY_COLOR="white"
STAR_COLOR="black"
SELECTED_CONSTELLATIONS=None
RA_GRID_DEG=list(range(0,360,30))
DEC_GRID_DEG=[-60.0,-30.0,0.0,30.0,60.0]
ECL_LONG = list(range(0, 360, 30))
ECL_LAT =[-60.0, -30.0, 0.0, 30.0, 60.0]
GAL_LONG = list(range(0, 360, 30))
GAL_LAT =[-60.0, -30.0, 0.0, 30.0, 60.0]


In [ ]:
observer = Observer(
    location="La Ligua",
    time="2026-08-15 21:00",
)


In [ ]:
print("Hipparcos:",catalog_path("hipparcos"))
print("Western lines:",constellation_lines_path("western"))
print("IAU boundaries:",boundary_path("iau"))


In [ ]:
projection=StereographicProjection(radius=PROJECTION_RADIUS,flip_ew=FLIP_EAST_WEST)
sky=CelestialSphere(observer=observer)
stars=sky.add_stars(catalog="hipparcos",magnitude_limit=MAGNITUDE_LIMIT)
constellations=sky.add_constellations(system="western",selected=SELECTED_CONSTELLATIONS)
boundaries=sky.add_constellation_boundaries(boundaries="iau",constellations=SELECTED_CONSTELLATIONS)
boundaries.sample()
points=sky.add_points()
points.add_equatorial_pole(pole="visible",marker="+",label="SCP",size=120,color="white",zorder=layers.POINTS)
points.add_ecliptic_pole(pole="south",marker="+",label="SEP",size=80,color="darkorange",zorder=layers.POINTS)
points.add_ecliptic_keypoints(marker="+",size=70,color="cyan",zorder=layers.POINTS)
points.add_galactic_center(marker="+",label="GC",size=80,color="lightblue",zorder=layers.POINTS)


In [ ]:
for name,method in {
"stars.draw":stars.draw,
"constellations.draw_lines":constellations.draw_lines,
"constellations.draw_labels":constellations.draw_labels,
"boundaries.draw":boundaries.draw,
"points.draw":points.draw,
"sky.draw_equatorial_grid":sky.draw_equatorial_grid,
}.items():
    print(f"{name}{inspect.signature(method)}")


In [ ]:
fig,ax=plt.subplots(figsize=(15,15))
fig.patch.set_alpha(0.0)
ax.patch.set_alpha(0.0)
sky_disk=Circle((0,0),PROJECTION_RADIUS,edgecolor="black",facecolor=SKY_COLOR,linewidth=1.5,zorder=-1000)
ax.add_patch(sky_disk)
ax.set_aspect("equal")
ax.set_xlim(-1.04*PROJECTION_RADIUS,1.04*PROJECTION_RADIUS)
ax.set_ylim(-1.04*PROJECTION_RADIUS,1.04*PROJECTION_RADIUS)
ax.axis("off")
plt.close(fig)

In [ ]:
boundary_artists = boundaries.draw(
    ax=ax,
    projection=projection,
)
for artist in boundary_artists:
    artist.set_color("royalblue")
    artist.set_linewidth(0.5)
    artist.set_alpha(0.7)
    
print("Number of boundary segments:", len(boundary_artists))

In [ ]:
sky.draw_ecliptic(
    ax=ax,
    projection=projection,
    color="darkorange",
    linewidth=1.0,
    linestyle="-",
    alpha=0.9,
    zorder=layers.CURVES,
)

In [ ]:
sky.draw_galactic_plane(
    ax=ax,
    projection=projection,
    color="steelblue",
    linewidth=1.0,
    linestyle="--",
    alpha=0.9,
    zorder=layers.CURVES,
)

In [ ]:
plot_eq_grid = True
plot_ec_grid = False
plot_gal_grid = False

if plot_eq_grid :
    eq_artists=sky.draw_equatorial_grid(ax=ax,projection=projection,ra=RA_GRID_DEG,dec=DEC_GRID_DEG,color="white",linewidth=0.4,alpha=0.35,zorder=layers.CURVES)
    for artist in eq_artists:
        artist.set_color("darkgrey")
        artist.set_linewidth(0.6)
        artist.set_linestyle((0, (1, 2)))
        artist.set_alpha(0.8)

if plot_ec_grid :
    ec_artists = sky.draw_ecliptic_grid(
        ax=ax,
        projection=projection,
        longitude=ECL_LONG,
        latitude=ECL_LAT,
    )
    for artist in ec_artists:
        artist.set_color("darkorange")
        artist.set_linewidth(0.8)
        artist.set_linestyle((0, (1, 2)))
        artist.set_alpha(0.8)

if plot_ec_grid :
    gal_artists = sky.draw_galactic_grid(
    ax=ax,
    projection=projection,
    longitude=GAL_LONG,
    latitude=GAL_LAT,
    )

    for artist in gal_artists:
        artist.set_color("steelblue")
        artist.set_linewidth(0.8)
        artist.set_linestyle((0, (1, 2)))
        artist.set_alpha(0.8)

In [ ]:
params=inspect.signature(stars.draw).parameters
if "observer" in params or "obs" in params:
    star_artists=stars.draw(ax,observer,t,projection,color=STAR_COLOR,alt_min=0.0)
else:
    star_artists=stars.draw(ax=ax,projection=projection,color=STAR_COLOR)
print("Star artists:",star_artists)

In [ ]:
constellation_line_artists = constellations.draw_lines(
    ax=ax,
    projection=projection,
)
for artist in constellation_line_artists:
    artist.set_color("grey")
    artist.set_linewidth(0.7)
    
print(
    "Number of constellation-line segments:",
    len(constellation_line_artists),
)
print("Lines after constellation lines:", len(ax.lines))

In [ ]:
import inspect

print(inspect.signature(constellations.draw_labels))
print(inspect.getsource(constellations.draw_labels))

In [ ]:
constellation_label_artists = constellations.draw_labels(
    ax=ax,
    selected=SELECTED_CONSTELLATIONS,
    min_stars=3,
    radial_cut=3.0,
    fontsize=10,
    color="white",
    alpha=0.85,
    outward_offset=0.04,
    zorder=layers.LABELS,
)
for artist in constellation_label_artists:
    artist.set_color("grey")
    artist.set_fontsize(10)
    artist.set_alpha(0.85)
print(
    "Number of constellation labels:",
    len(constellation_label_artists),
)
print("Texts after labels:", len(ax.texts))

In [ ]:
point_artists=points.draw(ax=ax,projection=projection)
print("Point artists:", point_artists)
print("Collections after points:", len(ax.collections))
print("Texts after points:", len(ax.texts))


In [ ]:
ax.set_title("Southern sky — 33° S — 15 August 2026, 21:00 Chile",fontsize=14,pad=18)
print("Axes lines:",len(ax.lines))
print("Axes patches:",len(ax.patches))
print("Axes collections:",len(ax.collections))
print("Axes texts:",len(ax.texts))

fig.text(
    0.99,
    0.01,
    f"Generated with Wenu {wenu.__version__}",
    ha="right",
    va="bottom",
    fontsize=6,
    alpha=0.7,
)
display(fig)


In [ ]:
OUTPUT_FILE=Path("carta_33S_2026-08-15_21h_explicit_wenu.png")
fig.savefig(OUTPUT_FILE,transparent=True,dpi=600,bbox_inches="tight",pad_inches=0.05)
print("Saved:",OUTPUT_FILE.resolve())


In [ ]:
print(inspect.getsource(type(stars)))

In [ ]:
stars.hip_df["magnitude"].describe()
